# FrontierMem Seed 17 — Stage B: evaluate BOTH existing checkpoints (NO training)

### Purpose

This notebook contains **zero training commands**. Attach these two Kaggle Inputs (as ZIP files or extracted directory trees):

- `FrontierMem_v4_Pointwise_Seed17_Checkpoint.zip` (already produced)
- `FrontierMem_Caid_Seed17_Complete_Training_Checkpoint.zip` (from Stage A)

Both methods are evaluated on the *same* RPEval Explicit, RPEval Implicit and BenchPreS sets; pair-direction evaluation uses **held-out pairs only** (no wasted GPU passes over the training pairs).

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json
REPO_URL='https://github.com/MohammadAmanlou/FrontierMem.git'
REPO_COMMIT='fca62bf316fabd629edf78ea46a36620d331ffe2'
MODEL='Qwen/Qwen3-4B-Instruct-2507'
SEED=17
WORKING=Path('/kaggle/working'); PROJECT=WORKING/'FrontierMem'
RESULTS=WORKING/'frontiermem_seed17_no_repeat'

def run(cmd,cwd=None,check=True,env=None):
    print('\n$ '+str(cmd)+'\n',flush=True)
    return subprocess.run(cmd,shell=True,cwd=cwd,env=env,check=check)

if PROJECT.exists():
    shutil.rmtree(PROJECT)
token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    pass
if token:
    askpass=WORKING/'.fm_askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token";;\n*Password*) echo "$GITHUB_TOKEN";;\n*) echo "";;\nesac\n')
    askpass.chmod(0o700)
    env=os.environ.copy();env['GITHUB_TOKEN']=token
    env['GIT_ASKPASS']=str(askpass);env['GIT_TERMINAL_PROMPT']='0'
    try: run(f'git clone {REPO_URL} {PROJECT}',env=env)
    finally: askpass.unlink(missing_ok=True)
else:
    run(f'git clone {REPO_URL} {PROJECT}')
run(f'git checkout {REPO_COMMIT}',cwd=PROJECT)
run(f'{sys.executable} -m pip uninstall -y torchao',check=False)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.57,<5" "peft>=0.20,<0.21" "bitsandbytes>=0.46" "accelerate>=1.0" "pandas>=2.2,<2.4" "scikit-learn>=1.4,<1.9"')
run(f'{sys.executable} -m pip install -q -r requirements-train.txt',cwd=PROJECT)
run(f'{sys.executable} -m pip install -q -e .',cwd=PROJECT)
import torch
assert torch.cuda.is_available(),'Enable GPU in Kaggle settings.'
print('GPU:',torch.cuda.get_device_name(0))
RESULTS.mkdir(parents=True,exist_ok=True)
summary=json.loads((PROJECT/'data/external_processed/summary.json').read_text())
assert summary['n_atomic_examples']==12423
assert summary['by_source']['benchpres']==1950
assert sum(1 for ln in open(PROJECT/'data/external_processed/caid_pairwise_train.jsonl',encoding='utf-8') if ln.strip())==6336
print('Dataset assertions passed; pinned commit:', REPO_COMMIT)

### Install evaluator with independent Platt and conformal calibration groups

In [ ]:
SCRIPT = '#!/usr/bin/env python\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, f1_score, brier_score_loss\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\n\nfrom frontiermem.external_data import load_jsonl, normalize_rpeval_native_label\nfrom frontiermem.contrast_sets import load_pairs\nfrom frontiermem.identifiability import SplitConformalAbstainer, selective_metrics\n\nLABELS = ["IGNORE", "SUPPORT", "DOMINATE"]\nLABEL2ID = {"IGNORE": 0, "SUPPORT": 1, "DOMINATE": 2}\nID2LABEL = {v: k for k, v in LABEL2ID.items()}\n\n\ndef format_row(row: dict) -> str:\n    parts = [f"[PREFERENCE]\\n{row.get(\'preference\', \'\')}"]\n    history = str(row.get("history", ""))\n    context = str(row.get("context", ""))\n    if history.strip():\n        parts.append(f"[HISTORY]\\n{history}")\n    if context.strip():\n        parts.append(f"[CONTEXT]\\n{context}")\n    parts.append(f"[QUERY]\\n{row.get(\'query\', \'\')}")\n    parts.append(\n        "[TASK]\\nJudge the preference-query relation using the native RPEval "\n        "labels IGNORE, SUPPORT, or DOMINATE."\n    )\n    parts.append("[DECISION]")\n    return "\\n".join(parts)\n\n\ndef app_score_np(z):\n    m = np.maximum(z[:, 1], z[:, 2])\n    lse = m + np.log(np.exp(z[:, 1] - m) + np.exp(z[:, 2] - m))\n    return lse - z[:, 0]\n\n\ndef softmax(x):\n    x = x - x.max(axis=1, keepdims=True)\n    e = np.exp(x)\n    return e / e.sum(axis=1, keepdims=True)\n\n\ndef native_label(row):\n    if not str(row.get("source", "")).startswith("rpeval"):\n        return None\n    try:\n        return normalize_rpeval_native_label(row.get("source_label", ""))\n    except ValueError:\n        return None\n\n\ndef load_model(checkpoint):\n    from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig\n    from peft import PeftConfig, PeftModel\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Evaluation requires CUDA in this v4 notebook.")\n\n    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    peft_cfg = PeftConfig.from_pretrained(checkpoint)\n    qconfig = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=compute_dtype,\n    )\n    base = AutoModelForSequenceClassification.from_pretrained(\n        peft_cfg.base_model_name_or_path,\n        num_labels=3,\n        id2label=ID2LABEL,\n        label2id=LABEL2ID,\n        quantization_config=qconfig,\n        device_map={"": 0},\n        dtype=compute_dtype,\n        trust_remote_code=True,\n        attn_implementation="sdpa",\n    )\n    model = PeftModel.from_pretrained(base, checkpoint)\n    model.config.use_cache = False\n    return model\n\n\n@torch.no_grad()\ndef logits_rows(model, tokenizer, rows, batch_size, max_length):\n    logits_all = []\n    model.eval()\n    device = torch.device("cuda:0")\n    for start in range(0, len(rows), batch_size):\n        chunk = rows[start:start + batch_size]\n        toks = tokenizer(\n            [format_row(r) for r in chunk],\n            padding=True,\n            truncation=True,\n            max_length=max_length,\n            return_tensors="pt",\n        )\n        toks = {k: v.to(device) for k, v in toks.items()}\n        logits_all.append(model(**toks).logits.float().cpu().numpy())\n    return np.concatenate(logits_all, axis=0)\n\n\ndef aar_mr(gold, pred):\n    gold = np.asarray(gold)\n    pred = np.asarray(pred)\n    pos = gold == "APPLY"\n    neg = gold == "IGNORE"\n    aar = float(np.mean(pred[pos] == "APPLY")) if pos.any() else float("nan")\n    mr = float(np.mean(pred[neg] == "APPLY")) if neg.any() else float("nan")\n    return aar, mr\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--checkpoint", required=True)\n    p.add_argument("--examples", default="data/external_processed/all_applicability_examples.jsonl")\n    p.add_argument("--pairs", default="data/external_processed/query_preference_contrast_pairs.jsonl")\n    p.add_argument("--output-dir", required=True)\n    p.add_argument("--alpha", type=float, default=0.10)\n    p.add_argument("--batch-size", type=int, default=4)\n    p.add_argument("--max-length", type=int, default=1024)\n    args = p.parse_args()\n\n    from transformers import AutoTokenizer\n\n    rows = [r for r in load_jsonl(args.examples) if r.get("action") in {"APPLY", "IGNORE"}]\n    calibration = [r for r in rows if r.get("usage") == "calibration"]\n    evaluation = [r for r in rows if r.get("usage") in {"eval", "eval_only"}]\n    if not calibration:\n        raise SystemExit("No leakage-safe calibration split found.")\n\n    tokenizer = AutoTokenizer.from_pretrained(args.checkpoint, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n\n    model = load_model(args.checkpoint)\n    model.config.pad_token_id = tokenizer.pad_token_id\n    if hasattr(model, "base_model") and hasattr(model.base_model, "config"):\n        model.base_model.config.pad_token_id = tokenizer.pad_token_id\n\n    cal_logits = logits_rows(model, tokenizer, calibration, args.batch_size, args.max_length)\n    cal_probs = softmax(cal_logits)\n    cal_score = app_score_np(cal_logits)\n    y_cal = np.asarray([1 if r["action"] == "APPLY" else 0 for r in calibration])\n\n    # IMPORTANT: Platt and conformal MUST use disjoint calibration groups.\n    # This avoids reusing the same labels to fit the calibrator and estimate qhat.\n    import random\n    group_keys = [str(r.get("group_id") or r.get("example_id")) for r in calibration]\n    all_groups = sorted(set(group_keys))\n    if len(all_groups) < 4:\n        raise RuntimeError("Insufficient calibration groups for group-disjoint split")\n    random.Random(1729).shuffle(all_groups)\n    platt_groups = set(all_groups[:max(1, len(all_groups) // 2)])\n    ix_platt = np.array([i for i, g in enumerate(group_keys) if g in platt_groups], dtype=int)\n    ix_conformal = np.array([i for i, g in enumerate(group_keys) if g not in platt_groups], dtype=int)\n    if min(len(ix_platt), len(ix_conformal)) == 0:\n        raise RuntimeError("Calibration split empty")\n    if len(set(y_cal[ix_platt].tolist())) < 2 or len(set(y_cal[ix_conformal].tolist())) < 2:\n        raise RuntimeError("Each calibration partition must contain both classes")\n    platt = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000, random_state=17)\n    platt.fit(cal_score[ix_platt].reshape(-1, 1), y_cal[ix_platt])\n    p_apply_conformal = platt.predict_proba(cal_score[ix_conformal].reshape(-1, 1))[:, 1]\n    abstainer = SplitConformalAbstainer(alpha=args.alpha).fit(\n        p_apply_conformal, y_cal[ix_conformal]\n    )\n\n    out = Path(args.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    (out / "calibration.json").write_text(\n        json.dumps({\n            "platt_coef": float(platt.coef_[0, 0]),\n            "platt_intercept": float(platt.intercept_[0]),\n            "conformal_alpha": args.alpha,\n            "conformal_qhat": float(abstainer.qhat),\n            "calibration_n": len(calibration),\n            "platt_fit_n": len(ix_platt),\n            "conformal_fit_n": len(ix_conformal),\n            "calibration_split": "group-disjoint, randomized seed 1729",\n            "calibration_brier": float(brier_score_loss(y_cal[ix_conformal], p_apply_conformal)),\n        }, indent=2),\n        encoding="utf-8",\n    )\n\n    metric_rows, pred_rows = [], []\n\n    for source in sorted({r["source"] for r in evaluation}):\n        part = [r for r in evaluation if r["source"] == source]\n        logits = logits_rows(model, tokenizer, part, args.batch_size, args.max_length)\n        probs = softmax(logits)\n        raw_p_apply = probs[:, 1] + probs[:, 2]\n        score = app_score_np(logits)\n        calibrated_p_apply = platt.predict_proba(score.reshape(-1, 1))[:, 1]\n\n        raw_forced = np.where(raw_p_apply >= 0.5, "APPLY", "IGNORE")\n        cal_forced = np.where(calibrated_p_apply >= 0.5, "APPLY", "IGNORE")\n        selective = np.asarray(abstainer.predict(calibrated_p_apply))\n        gold_binary = np.asarray([r["action"] for r in part])\n        y_binary = (gold_binary == "APPLY").astype(int)\n\n        raw_aar, raw_mr = aar_mr(gold_binary, raw_forced)\n        cal_aar, cal_mr = aar_mr(gold_binary, cal_forced)\n\n        record = {\n            "source": source,\n            "n": len(part),\n            "alpha": args.alpha,\n            "qhat": float(abstainer.qhat),\n            "raw_binary_accuracy": float(accuracy_score(gold_binary, raw_forced)),\n            "raw_binary_macro_f1": float(\n                f1_score(gold_binary, raw_forced, labels=["APPLY", "IGNORE"], average="macro", zero_division=0)\n            ),\n            "raw_AAR": raw_aar,\n            "raw_MR": raw_mr,\n            "raw_brier": float(brier_score_loss(y_binary, raw_p_apply)),\n            "cal_binary_accuracy": float(accuracy_score(gold_binary, cal_forced)),\n            "cal_binary_macro_f1": float(\n                f1_score(gold_binary, cal_forced, labels=["APPLY", "IGNORE"], average="macro", zero_division=0)\n            ),\n            "cal_AAR": cal_aar,\n            "cal_MR": cal_mr,\n            "cal_brier": float(brier_score_loss(y_binary, calibrated_p_apply)),\n            **{f"selective_{k}": v for k, v in selective_metrics(gold_binary, selective).items()},\n        }\n\n        native_gold = [native_label(r) for r in part]\n        native_idx = [i for i, y in enumerate(native_gold) if y is not None]\n        if native_idx:\n            native_true = [native_gold[i] for i in native_idx]\n            native_pred = [LABELS[int(logits[i].argmax())] for i in native_idx]\n            record["native_3way_accuracy"] = float(accuracy_score(native_true, native_pred))\n            record["native_3way_macro_f1"] = float(\n                f1_score(native_true, native_pred, labels=LABELS, average="macro", zero_division=0)\n            )\n            record["native_n"] = len(native_idx)\n\n        metric_rows.append(record)\n\n        for i, r in enumerate(part):\n            pred_rows.append({\n                "example_id": r["example_id"],\n                "source": source,\n                "gold_binary": r["action"],\n                "gold_native": native_label(r) or "",\n                "applicability_score": float(score[i]),\n                "p_ignore": float(probs[i, 0]),\n                "p_support": float(probs[i, 1]),\n                "p_dominate": float(probs[i, 2]),\n                "raw_p_apply": float(raw_p_apply[i]),\n                "calibrated_p_apply": float(calibrated_p_apply[i]),\n                "raw_forced_prediction": str(raw_forced[i]),\n                "cal_forced_prediction": str(cal_forced[i]),\n                "selective_prediction": str(selective[i]),\n                "native_prediction": LABELS[int(logits[i].argmax())],\n                "preference": r["preference"],\n                "query": r["query"],\n            })\n\n    pair_metric_rows = []\n    pair_path = Path(args.pairs)\n    if pair_path.exists():\n        pairs = load_pairs(pair_path)\n\n        def side_rows(part, prefix):\n            return [{\n                "preference": p.get(f"{prefix}_preference", ""),\n                "history": p.get(f"{prefix}_history", ""),\n                "context": p.get(f"{prefix}_context", ""),\n                "query": p.get(f"{prefix}_query", ""),\n            } for p in part]\n\n        for usage in sorted({str(p.get("usage", "")) for p in pairs}):\n            usage_rows = [p for p in pairs if str(p.get("usage", "")) == usage]\n            for source in sorted({str(p.get("source", "")) for p in usage_rows}):\n                part = [p for p in usage_rows if str(p.get("source", "")) == source]\n                if not part:\n                    continue\n                lp = logits_rows(model, tokenizer, side_rows(part, "positive"), args.batch_size, args.max_length)\n                ln = logits_rows(model, tokenizer, side_rows(part, "negative"), args.batch_size, args.max_length)\n                margin = app_score_np(lp) - app_score_np(ln)\n                pair_metric_rows.append({\n                    "usage": usage,\n                    "source": source,\n                    "n_pairs": len(part),\n                    "direction_accuracy": float((margin > 0).mean()),\n                    "mean_margin": float(margin.mean()),\n                    "median_margin": float(np.median(margin)),\n                })\n\n    pd.DataFrame(metric_rows).to_csv(out / "metrics.csv", index=False)\n    pd.DataFrame(pred_rows).to_csv(out / "predictions.csv", index=False)\n    pd.DataFrame(pair_metric_rows).to_csv(out / "pair_metrics_by_split_source.csv", index=False)\n\n    print(pd.DataFrame(metric_rows).to_string(index=False))\n    print("\\nPAIR METRICS")\n    print(pd.DataFrame(pair_metric_rows).to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n'
p=(PROJECT/'scripts'/'v4_1_evaluate_disjoint_cal.py')
p.write_text(SCRIPT,encoding='utf-8')
compile(SCRIPT,str(p),'exec')
print('Script installed:',p)

### Restore Pointwise and CAID checkpoints from Kaggle Inputs. No training is performed.

In [ ]:
import zipfile, shutil, hashlib
input_root=Path('/kaggle/input')

def restore_checkpoint(zip_names, prefix, target, parent_marker):
    zip_candidates=[]
    for zname in zip_names:
        zip_candidates += sorted(input_root.rglob(zname))
    if zip_candidates:
        source=zip_candidates[0]
        print('ZIP source:',source)
        with zipfile.ZipFile(source) as z:
            items=[n for n in z.namelist() if n.startswith(prefix) and not n.endswith('/')]
            assert prefix+'best/adapter_model.safetensors' in items, 'Adapter missing in input ZIP'
            for n in items:
                rel=n[len(prefix):]
                destination=target/rel
                destination.parent.mkdir(parents=True,exist_ok=True)
                with z.open(n) as src, destination.open('wb') as dst:
                    shutil.copyfileobj(src,dst,1024*1024)
    else:
        # Kaggle may have extracted ZIP contents at Dataset creation time.
        matches=[]
        for p in input_root.rglob('adapter_model.safetensors'):
            if parent_marker in p.as_posix() and p.parent.name=='best':
                matches.append(p)
        if not matches:
            raise FileNotFoundError(
                'Input not found. Attach '+repr(zip_names)+' as Kaggle Input. '
                'Do not retrain the existing models.'
            )
        base=matches[0].parent.parent
        print('Extracted source directory:',base)
        shutil.copytree(base,target,dirs_exist_ok=True)
    assert (target/'best'/'adapter_model.safetensors').exists()
    assert (target/'best'/'adapter_config.json').exists()
    assert (target/'best'/'best_meta.json').exists()
    print('Restored:',target)

pointwise_train=RESULTS/'pointwise'/'train'
caid_train=RESULTS/'caid'/'train'
restore_checkpoint(
    ['FrontierMem_v4_Pointwise_Seed17_Checkpoint.zip','results (8).zip'],
    'frontiermem_v4_final/seed_17/pointwise/train/',
    pointwise_train,
    'pointwise/train',
)
restore_checkpoint(
    ['FrontierMem_Caid_Seed17_Complete_Training_Checkpoint.zip'],
    'frontiermem_seed17_no_repeat/caid/train/',
    caid_train,
    'caid/train',
)
meta_pw=json.loads((pointwise_train/'best'/'best_meta.json').read_text())
meta_ca=json.loads((caid_train/'best'/'best_meta.json').read_text())
assert meta_pw['epoch']==1
assert abs(meta_pw['dev_binary_macro_f1']-0.9929906159414357)<1e-9
assert (caid_train/'training_summary.json').exists(),'CAID training completion summary is required.'
assert json.loads((caid_train/'training_summary.json').read_text())['best_epoch']==meta_ca['epoch']
print('Pointwise best epoch',meta_pw['epoch'],'CAID best epoch',meta_ca['epoch'])

### Build the held-out pair evaluation file (no generated text, no retraining)

In [ ]:
from collections import Counter
all_pairs=PROJECT/'data/external_processed/query_preference_contrast_pairs.jsonl'
heldout=PROJECT/'data/external_processed/eval_only_contrast_pairs.jsonl'
counts=Counter()
with all_pairs.open('r',encoding='utf-8') as f, heldout.open('w',encoding='utf-8') as out:
    for line in f:
        if not line.strip(): continue
        row=json.loads(line)
        if row.get('usage') in {'eval','eval_only'}:
            out.write(line)
            counts[(row.get('source'),row.get('usage'))]+=1
assert sum(counts.values())>1000
print('Held-out pairs for evaluation:',dict(counts))
print('Training pairs omitted from pair evaluation for efficiency.')

### Evaluate saved Pointwise checkpoint. This runs inference ONLY.

In [ ]:
pw_eval=RESULTS/'pointwise'/'eval'
if (pw_eval/'metrics.csv').exists() and (pw_eval/'predictions.csv').exists() and (pw_eval/'pair_metrics_by_split_source.csv').exists():
    print('Pointwise evaluation already complete in this session: SKIP')
else:
    run(
        f'{sys.executable} scripts/v4_1_evaluate_disjoint_cal.py '
        f'--checkpoint {pointwise_train/"best"} '
        '--examples data/external_processed/all_applicability_examples.jsonl '
        '--pairs data/external_processed/eval_only_contrast_pairs.jsonl '
        f'--output-dir {pw_eval} '
        '--alpha 0.10 --batch-size 4 --max-length 1024',
        cwd=PROJECT,
    )
print('POINTWISE INFERENCE DONE. This used no GPU training.')

### Evaluate saved CAID checkpoint. This runs inference ONLY.

In [ ]:
caid_eval=RESULTS/'caid'/'eval'
if (caid_eval/'metrics.csv').exists() and (caid_eval/'predictions.csv').exists() and (caid_eval/'pair_metrics_by_split_source.csv').exists():
    print('CAID evaluation already complete in this session: SKIP')
else:
    run(
        f'{sys.executable} scripts/v4_1_evaluate_disjoint_cal.py '
        f'--checkpoint {caid_train/"best"} '
        '--examples data/external_processed/all_applicability_examples.jsonl '
        '--pairs data/external_processed/eval_only_contrast_pairs.jsonl '
        f'--output-dir {caid_eval} '
        '--alpha 0.10 --batch-size 4 --max-length 1024',
        cwd=PROJECT,
    )
print('CAID INFERENCE DONE. This used no GPU training.')

### Verify coverage and package outputs. Checkpoints are archived separately.

In [ ]:
import pandas as pd, zipfile, hashlib
metric_frames=[]; pair_frames=[]
for method in ['pointwise','caid']:
    d=RESULTS/method/'eval'
    for name in ['metrics.csv','predictions.csv','pair_metrics_by_split_source.csv','calibration.json']:
        assert (d/name).exists(),f'Missing {d/name}'
    m=pd.read_csv(d/'metrics.csv')
    assert set(m['source'])=={'rpeval_explicit','rpeval_implicit','benchpres'},m['source'].tolist()
    for source,n in [('rpeval_explicit',953),('rpeval_implicit',953),('benchpres',1950)]:
        assert len(pd.read_csv(d/'predictions.csv').query('source == @source'))==n
    m.insert(0,'method',method);m.insert(0,'seed',17);metric_frames.append(m)
    pair=pd.read_csv(d/'pair_metrics_by_split_source.csv')
    assert set(pair['usage']).issubset({'eval','eval_only'})
    pair.insert(0,'method',method);pair.insert(0,'seed',17);pair_frames.append(pair)

allm=pd.concat(metric_frames,ignore_index=True)
allp=pd.concat(pair_frames,ignore_index=True)
allm.to_csv(RESULTS/'seed17_metrics_both_models.csv',index=False)
allp.to_csv(RESULTS/'seed17_heldout_pair_metrics_both_models.csv',index=False)
provenance={
 'seed':17,'repo_commit':REPO_COMMIT,'backbone':MODEL,
 'pointwise_training':'previously saved; partial run ended after epoch 2, best epoch 1; NOT retrained',
 'caid_training':'new completed Stage A; best checkpoint saved',
 'inference':'RPEval Explicit, RPEval Implicit, BenchPreS',
 'pair_eval':'held-out only, group-preserving benchmark-native pairs',
 'calibration':'Platt and conformal on separate groups of RPEval generation calibration split',
 'training_comparison_caveat':'pointwise used earlier microbatch=1/accum=16; CAID uses 2 pairs/microbatch, accum=4; cannot assert identical microbatch and full budget',
 'test_sets_used_for_training_or_checkpoint_selection':False,
 'no_github_push':True,
}
(RESULTS/'seed17_provenance.json').write_text(json.dumps(provenance,indent=2))
display(allm)
display(allp)
out=WORKING/'FrontierMem_Seed17_Complete_Evaluation_Results.zip'
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED,compresslevel=2) as z:
    for p in RESULTS.rglob('*'):
        if p.is_file() and p.name!='adapter_model.safetensors':
            z.write(p,p.relative_to(RESULTS.parent))
with zipfile.ZipFile(out) as z: assert z.testzip() is None
print('DOWNLOAD FINAL EVALUATION ZIP:',out)
print('Keep BOTH checkpoint ZIPs separately; adapter weights excluded from this evaluation bundle.')

### Completion criteria

**Operationally complete:** two saved checkpoints, CAID `training_summary.json`, both models have `metrics.csv`, `predictions.csv`, `pair_metrics_by_split_source.csv`, `calibration.json`, and the results bundle above.

**Scientific caveat:** original Pointwise training did not reach its planned patience/max-epoch completion and had a different microbatch schedule. This one-seed comparison is valuable but must be labeled carefully; strict matched multi-seed final results are a separate milestone.